In [3]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.utils import resample
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Load clean data — skip entire pipeline
X_train = pd.read_parquet('acrs_data/X_train.parquet')
X_val   = pd.read_parquet('acrs_data/X_val.parquet')
X_test  = pd.read_parquet('acrs_data/X_test.parquet')

y_train_arr = np.load('acrs_data/y_train.npy')
y_val_arr   = np.load('acrs_data/y_val.npy')
y_test_arr  = np.load('acrs_data/y_test.npy')

print(f"Loaded successfully | X_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}")

Loaded successfully | X_train: (449858, 83) | X_val: (375220, 83) | X_test: (504194, 83)


In [5]:
# Sample 100k for speed
X_sample, y_sample = resample(X_train, y_train_arr, n_samples=100000,
                               random_state=42, stratify=y_train_arr)

print("Tuning XGBoost...")
def xgb_objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 500),
        'max_depth':        trial.suggest_int('max_depth', 3, 8),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.2),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'eval_metric': 'auc', 'random_state': 42, 'n_jobs': -1
    }
    m = XGBClassifier(**params)
    m.fit(X_sample, y_sample)
    return roc_auc_score(y_val_arr, m.predict_proba(X_val)[:,1])

xgb_study = optuna.create_study(direction='maximize')
xgb_study.optimize(xgb_objective, n_trials=30)
print(f"XGBoost best AUC: {xgb_study.best_value:.4f}")
print(f"Best params: {xgb_study.best_params}\n")

print("Tuning LightGBM...")
def lgbm_objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 500),
        'max_depth':        trial.suggest_int('max_depth', 3, 8),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.2),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'random_state': 42, 'n_jobs': -1, 'verbose': -1
    }
    m = LGBMClassifier(**params)
    m.fit(X_sample, y_sample)
    return roc_auc_score(y_val_arr, m.predict_proba(X_val)[:,1])

lgbm_study = optuna.create_study(direction='maximize')
lgbm_study.optimize(lgbm_objective, n_trials=30)
print(f"LightGBM best AUC: {lgbm_study.best_value:.4f}")
print(f"Best params: {lgbm_study.best_params}\n")

print("Tuning Random Forest...")
def rf_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 400),
        'max_depth':    trial.suggest_int('max_depth', 5, 15),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),
        'random_state': 42, 'n_jobs': -1
    }
    m = RandomForestClassifier(**params)
    m.fit(X_sample, y_sample)
    return roc_auc_score(y_val_arr, m.predict_proba(X_val)[:,1])

rf_study = optuna.create_study(direction='maximize')
rf_study.optimize(rf_objective, n_trials=30)
print(f"Random Forest best AUC: {rf_study.best_value:.4f}")
print(f"Best params: {rf_study.best_params}")

Tuning XGBoost...
XGBoost best AUC: 0.7808
Best params: {'n_estimators': 437, 'max_depth': 3, 'learning_rate': 0.057924907827330084, 'subsample': 0.731809199181686, 'colsample_bytree': 0.7694293140093678}

Tuning LightGBM...
LightGBM best AUC: 0.7799
Best params: {'n_estimators': 399, 'max_depth': 5, 'learning_rate': 0.025457718459080056, 'subsample': 0.9810498493449374, 'colsample_bytree': 0.6693774287694965}

Tuning Random Forest...
Random Forest best AUC: 0.7785
Best params: {'n_estimators': 381, 'max_depth': 11, 'max_features': 'log2'}


In [7]:
from scipy.stats import ks_2samp

# Train final models on full X_train with best params
print("Training final models on full training set...")

xgb_final = XGBClassifier(**xgb_study.best_params, eval_metric='auc', 
                            random_state=42, n_jobs=-1)
xgb_final.fit(X_train, y_train_arr)
print("XGBoost done")

lgbm_final = LGBMClassifier(**lgbm_study.best_params, 
                              random_state=42, n_jobs=-1, verbose=-1)
lgbm_final.fit(X_train, y_train_arr)
print("LightGBM done")

rf_final = RandomForestClassifier(**rf_study.best_params, 
                                   random_state=42, n_jobs=-1)
rf_final.fit(X_train, y_train_arr)
print("Random Forest done")

# Soft voting ensemble
xgb_prob  = xgb_final.predict_proba(X_test)[:,1]
lgbm_prob = lgbm_final.predict_proba(X_test)[:,1]
rf_prob   = rf_final.predict_proba(X_test)[:,1]
ensemble_prob = (xgb_prob + lgbm_prob + rf_prob) / 3

# Metrics
def gini(y, p): return 2 * roc_auc_score(y, p) - 1
def ks(y, p): return ks_2samp(p[y==1], p[y==0]).statistic
from sklearn.metrics import f1_score

print("\n" + "="*58)
print(f"{'Model':<22} {'AUC':>7} {'KS':>7} {'Gini':>7} {'F1':>7}")
print("="*58)
for name, prob in [('XGBoost', xgb_prob), ('LightGBM', lgbm_prob),
                   ('Random Forest', rf_prob), ('Ensemble L1', ensemble_prob)]:
    auc = roc_auc_score(y_test_arr, prob)
    k   = ks(y_test_arr, prob)
    g   = gini(y_test_arr, prob)
    f1  = f1_score(y_test_arr, (prob>=0.5).astype(int))
    print(f"{name:<22} {auc:>7.4f} {k:>7.4f} {g:>7.4f} {f1:>7.4f}")
print("="*58)
print("\nLayer 1 — Hybrid Ensemble COMPLETE")

Training final models on full training set...
XGBoost done
LightGBM done
Random Forest done

Model                      AUC      KS    Gini      F1
XGBoost                 0.7438  0.3448  0.4876  0.2401
LightGBM                0.7409  0.3413  0.4818  0.2330
Random Forest           0.7374  0.3342  0.4747  0.2052
Ensemble L1             0.7424  0.3433  0.4848  0.2189

Layer 1 — Hybrid Ensemble COMPLETE


In [9]:
import pickle

with open('acrs_data/xgb_final.pkl', 'wb') as f: pickle.dump(xgb_final, f)
with open('acrs_data/lgbm_final.pkl', 'wb') as f: pickle.dump(lgbm_final, f)
with open('acrs_data/rf_final.pkl', 'wb') as f:   pickle.dump(rf_final, f)

np.save('acrs_data/ensemble_prob_test.npy', ensemble_prob)

print("Models saved — Layer 1 completed")

Models saved — Layer 1 completed
